# 05 · Training the two-tower finder

Step 1 (notebook 04) turned every article into 384 numbers with MiniLM. Now we train the two towers on Nov 9–13 clicks and compare against trending on Nov 14.

- **User tower:** average the vectors of your last 50 clicked articles, then a small trainable layer → 128 numbers.
- **Item tower:** the article's MiniLM vector through its own small layer → 128 numbers.
- **Score:** how close the two are (1 = same direction, 0 = unrelated).

**Training signal:** every click on Nov 9–13. The clicked article is pulled toward the user; 4 articles skipped on the same page and the other clicks in the batch are pushed away.

**Evaluation:** same as notebook 03: pool of 6,141 articles live on Nov 14, top 100 per bucket (500 total), Recall@K and NDCG@10.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import torch

from src.data import BUCKETS, explode_impressions, load_behaviors, load_news, split_by_time
from src.finder import hourly_clicks, load_embeddings
from src.metrics import evaluate
from src.two_tower import (
    MODEL_PATH, TwoTower, device, history_matrix, popularity_table, top_k_with_quotas, train,
    training_examples, user_vectors,
)

dev = device()
print("Running on:", dev)

## Prepare the data

In [ ]:
news = load_news()
news_ids, vectors = load_embeddings()
index_of = {n: i for i, n in enumerate(news_ids)}
bucket_of = news.dropna(subset=["bucket"]).set_index("news_id").bucket.to_dict()

behaviors = load_behaviors()
train_b, val_b = split_by_time(behaviors)

row_of, hist = history_matrix(behaviors, index_of)  # each user's last 50 past clicks
examples = training_examples(train_b, index_of, row_of)  # one example per Nov 9–13 click

article_vectors = torch.as_tensor(vectors, device=dev)
history = torch.as_tensor(hist, device=dev)
print(f"{len(examples[0]):,} training clicks  |  {len(row_of):,} users  |  {(hist[:, 0] == -1).mean():.1%} with no history")

## Train

In [ ]:
torch.manual_seed(0)
model = TwoTower().to(dev)
losses = train(model, article_vectors, history, examples, epochs=3)
model.eval()
torch.save(model.state_dict(), MODEL_PATH)

print("Loss per epoch:", [round(l, 3) for l in losses])

## Set up validation (Nov 14)

We also build a **trending table**: for every hour and article, `log(1 + clicks in the previous 24 hours)`. Same rule as notebook 03, so nothing peeks into the future.

In [ ]:
val_shown = explode_impressions(val_b)
pool = sorted(set(val_shown.news_id) & set(bucket_of))
pool_idx = torch.as_tensor([index_of[n] for n in pool], device=dev)
pool_buckets = np.array([bucket_of[n] for n in pool], dtype=object)

val_imp = (
    val_shown[(val_shown.clicked == 1) & val_shown.news_id.isin(bucket_of)]
    .groupby(["impression_id", "user_id", "time"]).news_id.agg(list).reset_index()
)
val_hours = val_imp.time.dt.floor("h").values

hour_row, table = popularity_table(hourly_clicks(behaviors), np.concatenate([examples[3], val_hours]), index_of)
popularity = torch.as_tensor(table, device=dev)
val_hour_rows = torch.as_tensor([hour_row[h] for h in pd.to_datetime(val_hours)], device=dev)
val_user_rows = torch.as_tensor([row_of[u] for u in val_imp.user_id], device=dev)
print(f"Pool: {len(pool):,} articles  |  {len(val_imp):,} validation page views")

In [ ]:
def score(model, beta, return_rankings=False):
    """Final score = two-tower similarity + beta × trending. Returns mean metrics over all validation page views."""
    with torch.no_grad():
        users = user_vectors(model, history[val_user_rows], article_vectors)
        items = model.items(article_vectors[pool_idx])
        rankings = []
        for start in range(0, len(val_imp), 4096):
            rows = slice(start, start + 4096)
            s = users[rows] @ items.T + beta * popularity[val_hour_rows[rows]][:, pool_idx]
            rankings += [[pool[c] for c in row] for row in top_k_with_quotas(s, pool_buckets, BUCKETS).cpu().numpy()]
    metrics = evaluate(rankings, val_imp.news_id.tolist()).mean()
    return (metrics, rankings) if return_rankings else metrics

## Results

- **Two-tower alone** (beta = 0): only your taste.
- **Two-tower + trending**: your taste plus a small boost for what's hot. We try a few boost sizes.

In [ ]:
baseline = pd.read_csv("../results/finder_baseline_metrics.csv", index_col=0)

results = pd.concat([
    baseline.loc[["random", "trending + quotas"]].rename(index={"trending + quotas": "trending"}),
    pd.DataFrame({
        "two-tower alone": score(model, 0.0),
        "two-tower + trending (0.03)": score(model, 0.03),
        "two-tower + trending (0.05)": score(model, 0.05),
        "two-tower + trending (0.10)": score(model, 0.10),
    }).T,
]).round(3)
results

## Experiment: add trending *during* training?

Idea: if the trending boost is part of the score while training, the towers only need to learn what trending doesn't already explain. We skip Nov 9 here because "trending" is zero for every article on the first day (no earlier data).

In [ ]:
keep = pd.to_datetime(examples[3]) >= pd.Timestamp("2019-11-10")
example_hours = torch.as_tensor([hour_row[h] for h in pd.to_datetime(examples[3][keep])], device=dev)

torch.manual_seed(0)
aware = TwoTower().to(dev)
train(aware, article_vectors, history, tuple(x[keep] for x in examples), epochs=3,
      popularity=popularity, example_hours=example_hours, beta=0.02)
aware.eval()

pd.DataFrame({
    "normal training + trending (0.05)": results.loc["two-tower + trending (0.05)"],
    "trending-aware training + trending (0.05)": score(aware, 0.05),
}).T.round(3)

In [ ]:
# Save the comparison for the README
results.to_csv("../results/finder_metrics.csv")

## Takeaways

- **Two-tower alone loses badly to trending** (Recall@500 0.22 vs 0.78). It knows your *taste* but not what's *hot today*. Of the hundreds of football stories, people click the one everyone is talking about.
- **Training barely beats plain MiniLM.** Averaging raw MiniLM vectors of your history scores about the same (0.23). With taste-only scores, the towers have little extra to learn.
- **Taste + trending beats trending on every metric.** With a 0.05 boost: Recall@500 **0.81 vs 0.78**, and NDCG@10 **0.074 vs 0.048 (about +50%)**. Your top 10 now differs from everyone else's, which is exactly where personalization should show.
- **Adding trending during training was mixed:** slightly better top 10 (NDCG@10 0.081 vs 0.074) but lower Recall@500 (0.80 vs 0.81), and larger boosts during training were clearly worse. Recall is the finder's main job, so we keep the simple version: train on taste, add trending when scoring.
- **Caveat:** the 0.05 boost was chosen on the validation day, so these numbers are slightly optimistic. The locked test set gives the honest number at the end.

**Final finder:** two-tower similarity + 0.05 × trending, top 100 per bucket.